# 11 — Batched Qwen/CrafText rollout → hybrid PPO evidence

Эта тетрадка выполняет synchronous MVP path: параллельные CrafText среды, MegaPrompts, один Tunix sampler call на batch, `jax.vmap(CrafText.step)`, per-env replay и новый hybrid PPO evidence contract. Она не подменяет `RLCluster` trainer: её задача — проверить, что rollout уже отдаёт PPO-ready evidence shapes и masks.


In [ ]:
from pathlib import Path

import jax.numpy as jnp
import matplotlib.pyplot as plt

from tunix_craftext.batched_rollout import collect_batched_text_rollout, replays_from_batched_rollout
from tunix_craftext.config import load_mvp_config
from tunix_craftext.hybrid_rollout import (
    compute_masked_step_token_ppo_loss,
    hybrid_step_from_text_trajectory,
    hybrid_trajectory_from_steps,
    shaped_step_rewards_from_text_trajectory,
)
from tunix_craftext.prompts import MegaPromptRenderer
from tunix_craftext.replay import save_replay
from tunix_craftext.runtime import build_craftext_runtime
from tunix_craftext.text_trajectory import text_trajectory_from_replay
from tunix_craftext.tunix_adapter import QwenTunixBackend


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run inside tunix-craftext.')

SNAPSHOT = ROOT / 'artifacts/models/qwen25-05b-instruct'
if not SNAPSHOT.is_dir():
    raise FileNotFoundError(f'No explicit local snapshot: {SNAPSHOT}')

config = load_mvp_config(ROOT / 'configs/mvp/qwen_craftext.yaml')
runtime = build_craftext_runtime(config)
BATCH_SIZE, HORIZON, MAX_NEW_TOKENS = 2, 8, 8
INVALID_ACTION_PENALTY = -0.2
print('actions:', runtime.actions.labels)
print('invalid action penalty:', INVALID_ACTION_PENALTY)


In [ ]:
rollout = collect_batched_text_rollout(
    runtime.adapter,
    MegaPromptRenderer(config.prompt.template),
    QwenTunixBackend(SNAPSHOT, cache_size=2048, seed=config.run.seed),
    actions=runtime.actions,
    batch_size=BATCH_SIZE,
    horizon=HORIZON,
    seed=config.run.seed,
    goal='Stay alive, obey the scenario instruction, and choose only legal actions.',
    max_new_tokens=MAX_NEW_TOKENS,
    invalid_action='fallback',
    fallback_action_id=runtime.actions.index_of('NOOP'),
)

for t, (decision, reset_mask) in enumerate(zip(rollout.decisions, rollout.reset_after_step, strict=True)):
    print('step', t, 'actions', [d.label for d in decision.actions], 'resets', reset_mask.tolist(), 'rewards', decision.transition.reward.tolist())


In [ ]:
replays = replays_from_batched_rollout(
    rollout,
    config_path='configs/mvp/qwen_craftext.yaml',
    commit='notebook',
    backend='tunix-single-device:Qwen',
)
output_dir = ROOT / 'artifacts/trajectories/end-to-end-batched-qwen'
for env_index, replay in enumerate(replays):
    path = output_dir / f'env-{env_index}.json'
    save_replay(path, replay)
    print('saved', path, 'steps=', len(replay.steps), 'fallbacks=', sum(step.fallback_used for step in replay.steps))


In [ ]:
hybrid_steps = []
for env_index, replay in enumerate(replays):
    batch = text_trajectory_from_replay(replay)
    values = jnp.zeros((batch.token_ids.shape[0],), dtype=jnp.float32)
    hybrid_step = hybrid_step_from_text_trajectory(batch, values=values)
    trajectory = hybrid_trajectory_from_steps([hybrid_step])
    shaped_rewards = shaped_step_rewards_from_text_trajectory(
        batch, invalid_action_penalty=INVALID_ACTION_PENALTY
    )
    advantages = shaped_rewards - hybrid_step.values
    actor_loss_tokens = int(jnp.sum(hybrid_step.actor_loss_token_mask))
    if actor_loss_tokens:
        loss = compute_masked_step_token_ppo_loss(
            hybrid_step.actor_log_probs,
            batch.old_logprobs,
            hybrid_step.actor_loss_token_mask,
            advantages,
            hybrid_step.step_mask,
        )
        loss_value = float(loss)
    else:
        loss_value = None
    hybrid_steps.append(hybrid_step)
    print(
        'env', env_index,
        'tokens', int(jnp.sum(hybrid_step.generation_token_mask)),
        'actor_loss_tokens', actor_loss_tokens,
        'invalid_actions', batch.invalid_action.tolist(),
        'shaped_rewards', shaped_rewards.tolist(),
        'step_masks', trajectory.step_masks.tolist(),
        'loss', loss_value,
        'note', 'fallback-only evidence; actor loss skipped' if loss_value is None else 'ok',
    )


In [ ]:
actions = jnp.stack([jnp.asarray([item.action_id for item in decision.actions]) for decision in rollout.decisions])
rewards = jnp.stack([decision.transition.reward for decision in rollout.decisions])
figure, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(actions.T, aspect='auto', interpolation='nearest')
axes[0].set(title='actions [env, time]', xlabel='time', ylabel='env')
axes[1].imshow(rewards.T, aspect='auto', interpolation='nearest', cmap='RdYlGn')
axes[1].set(title='rewards [env, time]', xlabel='time')
figure.tight_layout()


Итог: notebook показывает полный сбор PPO-ready evidence до learner boundary. Для реального update этот same-shaped hybrid evidence передаётся в Tunix `AgenticPPOLearner`/`RLCluster`, где actor пересчитывает current logprobs, critic выдаёт values, а checkpoints принадлежат Tunix trainer roles.
